# STIC runtine validation

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from evaspa.stic import main

### Functions

In [ ]:
# Constants
sigma = 5.67e-8  # Stefan-Boltzmann constant
gamma = 0.67  # Psychrometric constant (hpa/K)
Rd = 286.9  # Gas constant for dry air
Rw = 461.5  # Gas constant for wet air
P = 101325.0  # atmosphere pressure (Pascal)
Cpw = 1846.0  # Specific heat of moist air
Cpd = 1005.0  # Specific heat of air at constant pressure (J/kg/K)
threshold = 0.01  # Threshold for termination
alfa0 = 1.26  # Priestley-Taylor coefficient
kPAR = 0.5
kRN = 0.6

In [ ]:
def f_pSYCHROMETRICS(TS, TA, TD, RH, P, Rd, Rw, Cpw, Cpd, sigma):
    esstar = 6.13753 * np.exp(
        (17.27 * TS) / (TS + 237.3)
    )  # saturation vapor pressure at surface temperature, TS (unit hPa)
    eastar = 6.13753 * np.exp(
        (17.27 * TA) / (TA + 237.3)
    )  # saturation vapor pressure at air temperature, TA (unit hPa)
    ea = (RH / 100) * (eastar)  # actual vapor pressure of air (unit hPa)

    DA = eastar - ea  # vapor pressure deficit of air (hPa)
    slopeTA = (
        4098.171 * eastar / (TA + 237.3) ** 2
    )  # slope of saturation vapor pressure versus air temperature (hPa)
    # TD = (
    #    237.3 * np.log(ea / 6.13753) / (17.27 - np.log(ea / 6.13753))
    # )  # dewpoint temperature (deg C)
    #TD[TD < -30] = np.nan
    DPD = TA - TD  # dewpoint depression

    #slopeTD = 4098.171 * ea / (TD + 237.3) ** 2
    #slopeTS = 4098.171 * ea / (TS + 237.3) ** 2
    slopeTD = 4098.171 * 6.13753 * np.exp(
        (17.27 * TD) / (TD + 237.3)
    ) / (TD + 237.3) ** 2
    slopeTS = 4098.171 * 6.13753 * np.exp(
        (17.27 * TS) / (TS + 237.3)
    ) / (TS + 237.3) ** 2
    slopeTaTd = (eastar - ea) / (
        TA - TD
    )  # slope of saturation vapor pressure at dewpoint temperature

    qref = (0.622 * ea * 100) / (101325 - (0.387 * ea * 100))
    rhoD = P / (Rd * (TA + 273.15))  # density of dry air
    rho = rhoD * ((1 + qref) / (1 + qref * (Rw / Rd)))  # density of air
    CP = qref * Cpw + (1 - qref) * Cpd  # specific heat of air

    gR = ((4 * sigma * (TA + 273.15) ** 3) / CP) * (
        (Rd * (TA + 273.15)) / P
    )  # radiative conductance (m/s)(momentarily not needed)

    dTS = TS - TA  # difference between TS and TA

    return (
        esstar,
        eastar,
        ea,
        DA,
        slopeTA,
        TD,
        DPD,
        slopeTD,
        slopeTS,
        rho,
        CP,
        gR,
        dTS,
    )

In [ ]:
def f_StateEQ(rho, CP, gamma, alfa, slope, PHI, e0, ea, e0star, M):
    # Aerodynamic conductance
    gB = (2 * PHI * alfa * slope * gamma) / (
        2 * CP * slope * e0 * rho
        - 2 * CP * slope * ea * rho
        - 2 * CP * ea * gamma * rho
        + CP * e0 * gamma * rho
        + CP * e0star * gamma * rho
        - CP * M * e0 * gamma * rho
        + CP * M * e0star * gamma * rho
    )

    # Surface conductance
    denominator = (
        CP * e0star**2 * gamma * rho
        - CP * e0**2 * gamma * rho
        - 2 * CP * slope * e0**2 * rho
        + 2 * CP * slope * ea * e0 * rho
        - 2 * CP * slope * ea * e0star * rho
        + 2 * CP * slope * e0 * e0star * rho
        + 2 * CP * ea * e0 * gamma * rho
        - 2 * CP * ea * e0star * gamma * rho
        + CP * M * e0**2 * gamma * rho
        + CP * M * e0star**2 * gamma * rho
        - 2 * CP * M * e0 * e0star * gamma * rho
    )
    denominator[denominator == 0] = 0.00001  # when e0star == e0
    gS = (
        -(
            2
            * (
                PHI * alfa * slope * ea * gamma
                - PHI * alfa * slope * e0 * gamma
            )
        )
        / denominator
    )

    # T0 - TA
    dT = (
        2 * slope * e0
        - 2 * slope * ea
        - 2 * ea * gamma
        + e0 * gamma
        + e0star * gamma
        - M * e0 * gamma
        + M * e0star * gamma
        + 2 * alfa * slope * ea
        - 2 * alfa * slope * e0
    ) / (2 * alfa * slope * gamma)

    # Evaporative fraction
    EF = -(2 * alfa * slope * ea - 2 * alfa * slope * e0) / (
        2 * slope * e0
        - 2 * slope * ea
        - 2 * ea * gamma
        + e0 * gamma
        + e0star * gamma
        - M * e0 * gamma
        + M * e0star * gamma
    )

    # Adjust the abnormal conductances
    gB[gB < 0] = 0.0001
    gB[gB > 0.2] = 0.2
    gS[gS < 0] = 0.0001
    gS[gS > 0.06] = 0.06

    # Maximum surface-air temperature difference rarely overpasses 20 degC
    dT[dT < -10] = -10
    dT[dT > 20] = 20
    EF[EF < 0] = 0.0001
    EF[EF > 1] = 1

    return (gB, gS, dT, EF)

In [ ]:
def f_SoilMoisture_INITIALIZE(
    gamma, slope, TS, TA, TD, dTS, RN, Lnet, fc, DA, eastar, ea, esstar
):
    """
    This function estimates the soil moisture availability
    (M or wetness, 0-1) based on thermal and meteorological
    information.
    However, this M will be treated as initial M,
    which will be later on estimated through iteration in the actual ET estimation loop to
    establish feedback between M and biophysical states
    """

    # TU computation (surface dewpoint temperature)
    s11 = (
        45.03 + 3.014 * TD + 0.05345 * TD**2 + 0.00224 * TD**3
    ) * 1e-2  # slope of SVP at TD (hpa/K)
    s22 = (esstar - ea) / (TS - TD)
    s33 = (
        45.03 + 3.014 * TS + 0.05345 * TS**2 + 0.00224 * TS**3
    ) * 1e-2  # slope of SVP at TS (hpa/K)
    s44 = (eastar - ea) / (TA - TD)
    s11 = (
        4098.171
        * 6.13753
        * np.exp((17.27 * TD) / (TD + 237.3))
        / (TD + 237.3) ** 2
    )
    s33 = (
        4098.171
        * 6.13753
        * np.exp((17.27 * TS) / (TS + 237.3))
        / (TS + 237.3) ** 2
    )

    # Surface dewpoint temperature (degC)
    T0D = (esstar - ea - s33 * TS + s11 * TD) / (s11 - s33)

    # Surface moisture (Msurf)
    Msurf = (s11 / s22) * (
        (T0D - TD) / (TS - TD)
    )  # Surface wetness  #######To be checked

    Msurf[Msurf > 1] = 0.9999
    Msurf[Msurf < 0] = 0.0001

    # Surface vapor pressure and deficit
    esurf = ea + Msurf * (esstar - ea)
    Dsurf = esurf - ea

    # Separating soil and canopy wetness to form a composite surface moisture
    Ms = Msurf.copy()
    Mcan = fc * Msurf
    Msoil = (1 - fc) * Msurf

    TdewIndex = (TS - T0D) / (
        TA - TD
    )  # TdewIndex > 1 signifies super dry condition
    Ep_PT = (1.26 * slope * RN) / (
        slope + gamma
    )  # Potential evaporation (Priestley-Taylor eqn.)

    # Surface wetness comes from the soil, vegetation contribution is neglegible
    Ms[(fc <= 0.25) & (TdewIndex < 1)] = Msoil[(fc <= 0.25) & (TdewIndex < 1)]
    Mcan[(fc <= 0.25) & (TdewIndex < 1)] = 0

    Ms[(fc <= 0.25) & (TA > 10) & (TD < 0) & (Lnet < -125)] = Msoil[
        (fc <= 0.25) & (TA > 10) & (TD < 0) & (Lnet < -125)
    ]
    Mcan[(fc <= 0.25) & (TA > 10) & (TD < 0) & (Lnet < -125)] = 0

    # Root zone moisture (Mrz)
    Mrz = (gamma * s11 * (T0D - TD)) / (
        slope * s33 * (TS - TD)
        + gamma * s44 * (TA - TD)
        - slope * s11 * (T0D - TD)
    )

    Mrz[Mrz > 1] = 0.9999
    Mrz[Mrz < 0] = 0.0001

    # Combine M to account for Hysteresis and initial estimation of surface vapor pressure
    M = Ms.copy()
    M[(Ep_PT > RN) & (dTS > 0)] = Mrz[(Ep_PT > RN) & (dTS > 0)]
    M[(Ep_PT > RN) & (fc <= 0.25)] = Mrz[(Ep_PT > RN) & (fc <= 0.25)]
    M[(Ep_PT > RN) & (Dsurf > DA)] = Mrz[(Ep_PT > RN) & (Dsurf > DA)]

    M[(fc <= 0.25) & (dTS > 0) & (TA > 10) & (TD < 0) & (Lnet < -125)] = Mrz[
        (fc <= 0.25) & (dTS > 0) & (TA > 10) & (TD < 0) & (Lnet < -125)
    ]
    M[(fc <= 0.25) & (dTS > 0) & (TA > 10) & (TD < 0) & (Dsurf > DA)] = Mrz[
        (fc <= 0.25) & (dTS > 0) & (TA > 10) & (TD < 0) & (Dsurf > DA)
    ]
    M[(Ep_PT < RN) & (fc <= 0.25) & (Dsurf > DA)] = Mrz[
        (Ep_PT < RN) & (fc <= 0.25) & (Dsurf > DA)
    ]

    es = ea + M * (esstar - ea)

    # vapor pressure deficit at surface
    Ds = esstar - es

    return (M, Mcan, Msoil, Ms, Mrz, s11, s22, s33, s44, es, T0D, Ds, TdewIndex)


def f_SoilMoisture_ITERATE(
    gamma,
    slope,
    s1,
    s2,
    s3,
    s4,
    TS,
    TA,
    dTS,
    TD,
    T0D,
    RN,
    Lnet,
    fc,
    DA,
    D0,
    eastar,
    ea,
    e0star,
    esstar,
):
    """
    This functions estimates the soil moisture availability (M) (or
    wetness) (value 0 to 1) based on thermal IR and meteorological
    information. However, this M will be treated as initial M, which will be
    later on estimated through iteration in the actual ET estimation loop to
    establish feedback between M and biophysical states
    """
    # Surface moisture (Msurf)
    k = (e0star - ea) / (esstar - ea)
    Msurf = (s1 / (k * s2)) * ((T0D - TD) / (TS - TD))  # Surface wetness

    Msurf[Msurf > 1] = 0.9999
    Msurf[Msurf < 0] = 0.0001

    # Separating soil and canopy wetness to form a composite surface moisture
    Ms = Msurf.copy()
    Mcan = fc * Msurf
    Msoil = (1 - fc) * Msurf

    TdewIndex = (TS - T0D) / (TA - TD)
    Ep_PT = (1.26 * slope * RN) / (
        slope + gamma
    )  # Potential evaporation (Priestley-Taylor eqn.)

    Ms[(fc <= 0.25) & (TdewIndex < 1)] = Msoil[(fc <= 0.25) & (TdewIndex < 1)]
    Ms[(fc <= 0.25) & (TA > 10) & (TD < 0) & (Lnet < -125)] = Msoil[
        (fc <= 0.25) & (TA > 10) & (TD < 0) & (Lnet < -125)
    ]

    Mcan[(fc <= 0.25) & (TdewIndex < 1)] = 0
    Mcan[(fc <= 0.25) & (TA > 10) & (TD < 0) & (Lnet < -125)] = 0

    # Rootzone moisture (Mrz)
    Mrz = (gamma * s1 * (T0D - TD)) / (
        slope * s3 * (TS - TD)
        + gamma * s4 * (TA - TD)
        - slope * s1 * (T0D - TD)
    )

    Mrz[Mrz > 1] = 0.9999
    Mrz[Mrz < 0] = 0.0001

    TdewIndex = (TS - T0D) / (TA - TD)
    Ep_PT = (1.26 * slope * RN) / (
        slope + gamma
    )  # Potential evaporation (Priestley-Taylor eq.)

    # Combine M to account for Hysteresis and initial estimation of surface vapor pressure
    M = Ms.copy()
    M[(Ep_PT > RN) & (dTS > 0) & (fc <= 0.25) & (D0 > DA) & (TdewIndex < 1)] = (
        Mrz[
            (Ep_PT > RN)
            & (dTS > 0)
            & (fc <= 0.25)
            & (D0 > DA)
            & (TdewIndex < 1)
        ]
    )
    M[
        (Lnet < -125)
        & (dTS > 0)
        & (fc <= 0.25)
        & (D0 > DA)
        & (TA > 10)
        & (TD < 0)
    ] = Mrz[
        (Lnet < -125)
        & (dTS > 0)
        & (fc <= 0.25)
        & (D0 > DA)
        & (TA > 10)
        & (TD < 0)
    ]

    return (M, Ms, Mcan, Msoil, Mrz)

In [ ]:
def f_G_PHI_actualsurface(RN, RNsoil, ttSEC, M):
    cgMIN = 0.05  # for wet surface
    cgMAX = 0.35  # for dry surface, in water-controlled ecosystems
    #     cgMAX = 0.20 # for dry surface, in radiation-controlled ecosystems

    tgMIN = 74000  # for wet surface
    tgMAX = 100000  # for dry surface

    solNooN = 12 * 60 * 60
    tg0 = solNooN - ttSEC

    # Estimating GHF according to Santanello and Friedl (2003)
    cg = (1 - M) * cgMAX + M * cgMIN
    tg = (1 - M) * tgMAX + M * tgMIN

    # G = RN*cg*np.cos(2*np.pi*(tg0 + 10800)/tg)
    # G[RN < 0] = -G[RN < 0]

    G = (
        RNsoil * cg * np.cos(2 * np.pi * (tg0 + 10800) / tg)
    )  # Correction on 12/11/2021
    G[RNsoil < 0] = -G[RNsoil < 0]  # Correction on 12/11/2021

    PHI = RN - G

    return (cg, tg, tg0, G, PHI)

In [ ]:

def STIC(TS, TA, TD, RH, RN, Lnet, fc, LAI, ttSEC):
    # Mask for valid pixels
    mask = np.ones(TS.shape)
    mask[np.isnan(TS)] = 0

    ## Psychrometrics
    (
        esstar,
        eastar,
        ea,
        DA,
        slopeTA,
        _,
        _,
        s11,
        s33,
        rho,
        CP,
        gR,
        dTS,
    ) = f_pSYCHROMETRICS(TS, TA, TD, RH, P, Rd, Rw, Cpw, Cpd, sigma)
    #rho= np.array([1.183452]) 
    #CP = np.array([1014.621236])
    print(
        esstar,
        ea,
        DA,
        slopeTA,
        rho,
        CP,
    )

    # Initialize
    print("toto")
    print(
            gamma,
            slopeTA,
            TS,
            TA,
            TD,
            dTS,
            RN,
            Lnet,
            fc,
            DA,
            eastar,
            ea,
            esstar,
        )
    (M, Mcan, Msoil, Ms, Mrz, _, s22, _, s44, es, TSD, Ds, TdewIndex) = (
        f_SoilMoisture_INITIALIZE(
            gamma,
            slopeTA,
            TS,
            TA,
            TD,
            dTS,
            RN,
            Lnet,
            fc,
            DA,
            eastar,
            ea,
            esstar,
        )
    )
    print(s11, s22, s33, s44)
    print(M, Msoil, es, TSD, Ds)

    T0Dold = TSD.copy()
    e0star = esstar.copy()
    e0 = es.copy()
    slope = slopeTA.copy()

    # LAI = -np.log(1 - fc) / kPAR  # Correction on 12/11/2021
    RNsoil = RN * np.exp(-kRN * LAI)  # Correction on 12/11/2021

    
    print(RN, RNsoil, ttSEC, Msoil)
    (_, _, _, G, PHI) = f_G_PHI_actualsurface(
        RN, RNsoil, ttSEC, Msoil
    )  # Correction on 12/11/2021
    print(PHI,G)
    alfa = 1.26
    EpWET = (alfa * slope * PHI) / (slope + gamma)

    (gB, gS, dT, EF) = f_StateEQ(
        rho, CP, gamma, alfa, slope, PHI, e0, ea, e0star, M
    )
    print(gB, gS, dT, EF) 

    gB[gB < 0] = 0.0001
    gB[gB > 0.2] = 0.2
    gS[gS < 0] = 0.0001
    gS[gS > 0.06] = 0.06

    dT[dT < -10] = -10
    dT[dT > 20] = 20
    EF[EF < 0] = 0.0001
    EF[EF > 1] = 1

    gBB = gB.copy()
    gSS = gS.copy()
    dTT = dT.copy()
    EFF = EF.copy()

    T0 = dTT + TA
    slope00 = (e0star - ea) / (T0 - TD)
    # slope00 = slope.copy()

    # Calculate ET and H based on initial results from state eqs.
    # McNaughton and Jarvis (1986)
    omega1 = ((slope / gamma) + 1) / ((slope / gamma) + 1 + gBB / gSS)
    LHFeq1 = (PHI * (slope / gamma)) / ((slope / gamma) + 1)
    LHFimp1 = (CP * 0.0289644 / gamma) * gSS * 40 * DA
    LHF1 = omega1 * LHFeq1 + (1 - omega1) * LHFimp1

    LHF = LHF1.copy()
    SHF = (gamma * PHI * (1 + gBB / gSS) - rho * CP * gBB * DA) / (
        slope + gamma * (1 + gBB / gSS)
    )  # Deduced from the PM equation

    Ep = (slope * PHI + rho * CP * gBB * DA) / (
        slope + gamma
    )  # Potential evaporation (Penman)
    Tp = (slope * RN + rho * CP * gBB * DA) / (
        slope + gamma * (1 + M * gBB / gSS)
    )  # Potential transpiration

    Es = (1 - fc) * Ms * EpWET  # Soil evaporation
    Ei = Mcan * EpWET  # Canopy/interception evaporation
    Ei[(Ep > PHI) & (Ds - DA > 0) & (dTS > 0)] = 0

    EVAP = Es + Ei  # Total evaporation
    TRANSP = LHF - EVAP  # Transpiration

    # Iteration
    LHFold = LHF.copy()
    LHFnew = LHF.copy()
    error_LHF = np.ones(TS.shape) * 0.05
    steps = 0
    converged = np.asarray(
        np.zeros(TS.shape), dtype=bool
    )  # Create the status array

    # Create the variable arrays
    [D0, T0D, alfaN] = [
        np.zeros(TS.shape, np.float32) + np.nan for i in range(3)
    ]
    print(LHF)

    while np.nanmax(error_LHF) > threshold and steps < 15:
        index = np.where(converged)
        print("converged pixels: ", index[0].shape[0])

        # Selecting valid and non-converged pixels
        i = np.logical_and(mask, ~converged)

        # Re-estimate saturated vapor pressure at source/sink height
        e0star[i] = ea[i] + (gamma * LHFnew[i] * (gBB[i] + gSS[i])) / (
            rho[i] * CP[i] * gBB[i] * gSS[i]
        )
        e0star[np.logical_and(i, e0star < 0)] = esstar[
            np.logical_and(i, e0star < 0)
        ]
        e0star[np.logical_and(i, e0star > 250)] = 250

        # Re-estimate vapor pressure at source/sink height
        D0[i] = (
            (gBB[i] / gSS[i])
            * (gamma / (slope[i] + gamma * (1 + gBB[i] / gSS[i])))
            * (DA[i] + ((slope[i] * PHI[i]) / (rho[i] * CP[i] * gBB[i])))
        )
        D0[np.logical_and(i, D0 < 0)] = Ds[np.logical_and(i, D0 < 0)]

        e0[i] = e0star[i] - D0[i]
        e0[np.logical_and(i, e0 < 0)] = es[np.logical_and(i, e0 < 0)]
        e0[np.logical_and(i, e0 < ea)] = es[np.logical_and(i, e0 < ea)]
        e0[np.logical_and(i, e0 > e0star)] = es[np.logical_and(i, e0 > e0star)]

        # Re-estimate M (direct LST feedback into M computation)
        T0D[i] = TD[i] + (gamma * LHFnew[i]) / (
            rho[i] * CP[i] * gBB[i] * s11[i]
        )
        T0D[np.logical_and(i, T0D < TD)] = TD[np.logical_and(i, T0D < TD)]
        T0D[np.logical_and(i, T0D > TS)] = T0Dold[np.logical_and(i, T0D > TS)]

        (M[i], Ms[i], Mcan[i], Msoil[i], Mrz[i]) = f_SoilMoisture_ITERATE(
            gamma,
            slope[i],
            s11[i],
            s22[i],
            s33[i],
            s44[i],
            TS[i],
            TA[i],
            dTS[i],
            TD[i],
            T0D[i],
            RN[i],
            Lnet[i],
            fc[i],
            DA[i],
            D0[i],
            eastar[i],
            ea[i],
            e0star[i],
            esstar[i],
        )

        # Re-estimate PT coefficient
        alfaN[i] = (
            gSS[i]
            * slope00[i]
            * (T0[i] - TD[i])
            * (
                2 * slope[i]
                + 2 * gamma
                + gamma * (gBB[i] / gSS[i]) * (1 + M[i])
            )
        ) / (
            2
            * slope[i]
            * (
                gamma * (T0[i] - TA[i]) * (gBB[i] + gSS[i])
                + gSS[i] * slope00[i] * (T0[i] - TD[i])
            )
        )
        alfaN[np.logical_and(i, alfaN < 0)] = 1
        alfaN[np.logical_and(i, alfaN > 2)] = 2

        # Re-estimate net available energy
        (_, _, _, G[i], PHI[i]) = f_G_PHI_actualsurface(
            RN[i], RNsoil[i], ttSEC[i], Ms[i]
        )  # Correction on 15/11/2021

        # Re-estimate conductances and states

        (gBB[i], gSS[i], dTT[i], EFF[i]) = f_StateEQ(
            rho[i],
            CP[i],
            gamma,
            alfaN[i],
            slope[i],
            PHI[i],
            e0[i],
            ea[i],
            e0star[i],
            M[i],
        )
        print("coucou")
        print(
            rho[i],
            CP[i],
            gamma,
            alfaN[i],
            slope[i],
            PHI[i],
            e0[i],
            ea[i],
            e0star[i],
            M[i])
        print(gBB[i], gSS[i], dTT[i], EFF[i])

        gBB[np.logical_and(i, gBB < 0)] = 0.0001
        gBB[np.logical_and(i, gBB > 0.2)] = 0.2
        gSS[np.logical_and(i, gSS < 0)] = 0.0001
        gSS[np.logical_and(i, gSS > 0.06)] = 0.06
        dTT[np.logical_and(i, dTT < -10)] = -10
        dTT[np.logical_and(i, dTT > 20)] = 20
        EFF[np.logical_and(i, EFF < 0)] = 0.0001
        EFF[np.logical_and(i, EFF > 1)] = 1

        T0[i] = dTT[i] + TA[i]

        # Re-estimate LHF and SHF
        LHFnew[i] = (
            (rho[i] * CP[i] / gamma)
            * ((gBB[i] * gSS[i]) / (gBB[i] + gSS[i]))
            * (slope[i] * (T0[i] - TA[i]) + DA[i])
        )  # An alternative is to use the PM equation to calculate LHF

        index = np.logical_and.reduce((LHFnew < 0, LHFnew < PHI, i))
        LHFnew[index] = (
            rho[index]
            * CP[index]
            * gBB[index]
            * (e0[index] - ea[index])
            / gamma
        )

        SHF[i] = (
            (
                gamma * PHI[i] * (1 + gBB[i] / gSS[i])
                - rho[i] * CP[i] * gBB[i] * DA[i]
            )
            / (slope[i] + gamma * (1 + (gBB[i] / gSS[i])))
        )  # surface heat flux, SHF = PHI - LHF, PM equation is only used for calculating SHF in the current version of STIC

        slope00[i] = (
            (gamma * LHFnew[i]) / (rho[i] * CP[i] * gSS[i]) + (e0[i] - ea[i])
        ) / (T0[i] - TD[i])

        # Re-conduct ET partitioning
        Es[i] = (1 - fc[i]) * Ms[i] * EpWET[i]
        Ei[i] = Mcan[i] * EpWET[i]
        Ei[np.logical_and.reduce((i, Ep > RN, D0 - DA > 0, dTS > 0))] = 0
        EVAP[i] = Es[i] + Ei[i]
        TRANSP[i] = LHFnew[i] - EVAP[i]

        # Error
        error_LHF[i] = np.abs(LHFold[i] - LHFnew[i])
        LHFold[i] = LHFnew[i]
        steps = steps + 1

        converged[i] = error_LHF[i] < threshold

    LHF = LHFnew.copy()

    print("Number of iterations = ", steps)

    LE_STIC = LHF
    EF_STIC = LHF / (LHF + SHF)

    return (LE_STIC, EF_STIC, converged)

### Run

In [ ]:
STIC(
    TS=np.array([44.03]),
    TA=np.array([29.368835]),
    TD=np.array([10.886169]),
    RH=np.array([31.836037]),
    fc=np.array([0.17635]),
    LAI=np.array([0.02]),
    RN=np.array([514.07434]),
    Lnet=np.array([-186.61214]),
    ttSEC=np.array([32175.06206896552]),
)

In [ ]:
main.run_stic_model_pixel(
    ts=44.03,
    ta=29.368835,
    td=10.886169,
    rh=31.836037,
    fc=0.17635,
    lai=0.02,
    rn=514.07434,
    ln=-186.61214,
    local_time=32175.06206896552
)

In [ ]:
main.run_stic_model(
             np.array([[30, 30], [30, 30]]),
            np.array([[25, 25], [25, 25]]),
            np.array([[19, 19], [19, 19]]),
            np.array([[69.36, 69.36], [69.36, 69.36]]),
            np.array([[0.86, 0.86], [0.86, 0.86]]),
            np.array([[4, 4], [4, 4]]),
            np.array([[300, 300], [300, 300]]),
            np.array([[100, 100], [100, 100]]),
            np.array([[20000, 20000], [20000, 20000]]),
)